# 06: Time Series Forecasting & Customer Segmentation

This notebook covers two major areas of applied ML:

**Part A: Time Series Forecasting**
- Time series decomposition (trend, seasonality, residual)
- Moving averages and smoothing
- ARIMA models (AutoRegressive Integrated Moving Average)
- Facebook Prophet for business forecasting

**Part B: Clustering & Dimensionality Reduction**
- K-Means clustering
- DBSCAN (density-based clustering)
- PCA (Principal Component Analysis)
- RFM customer segmentation pipeline

---

**Key Idea:** Time series and clustering are both essential in business analytics. Time series tells you *what will happen next*; clustering tells you *who your customers are*.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

from shared.viz_utils import setup_style, save_fig
setup_style()
print("Setup complete")

---
# PART A: TIME SERIES FORECASTING

## Section 1: Time Series Basics

**What makes time series special?**
- **Temporal ordering matters** — shuffling rows destroys information (unlike tabular ML)
- **Autocorrelation** — values are correlated with their own past (today's sales depend on yesterday's)
- **Non-stationarity** — mean and variance can change over time

**Components of a time series:**
| Component | Description | Example |
|-----------|-------------|---------|
| **Trend** | Long-term direction | Sales growing 5% per year |
| **Seasonality** | Repeating patterns at fixed intervals | Holiday shopping spikes |
| **Residual** | Random noise after removing trend + seasonality | Day-to-day variation |

**Stationarity** — A series is stationary when its statistical properties (mean, variance) don't change over time. Most models (ARIMA) require stationarity. We test with the **Augmented Dickey-Fuller (ADF)** test and achieve it through **differencing**.

In [ ]:
# Generate synthetic daily sales data with trend + seasonality + noise
np.random.seed(42)
dates = pd.date_range('2021-01-01', '2024-12-31', freq='D')
n = len(dates)

# Build up the components
trend = np.linspace(100, 200, n)                          # steady upward trend
seasonal = 30 * np.sin(2 * np.pi * np.arange(n) / 365.25)  # yearly cycle
weekly = 10 * np.sin(2 * np.pi * np.arange(n) / 7)         # weekly cycle
noise = np.random.normal(0, 8, n)                           # random noise

sales = trend + seasonal + weekly + noise
ts = pd.DataFrame({'date': dates, 'sales': sales}).set_index('date')

# Plot the full series
ts.plot(figsize=(14, 4), title='Daily Sales (Synthetic)', legend=False, alpha=0.7)
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.show()

print(f"Date range: {ts.index.min().date()} to {ts.index.max().date()}")
print(f"Total observations: {len(ts)}")

In [ ]:
# Decompose the time series into trend, seasonal, and residual components
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(ts['sales'], period=365)

fig = result.plot()
fig.set_size_inches(14, 10)
fig.suptitle('Time Series Decomposition', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Decomposition separates the signal into interpretable components.")
print("This helps us understand WHAT is driving the data before we model it.")

In [ ]:
# Test for stationarity using the Augmented Dickey-Fuller test
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name=''):
    """Run ADF test and print results."""
    result = adfuller(series.dropna())
    print(f"ADF Test for {name}:")
    print(f"  Test Statistic: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.4f}")
    print(f"  Stationary?     {'YES' if result[1] < 0.05 else 'NO'} (p < 0.05)")
    print()

# Original series (has trend → likely non-stationary)
adf_test(ts['sales'], 'Original Sales')

# After differencing (removes trend)
adf_test(ts['sales'].diff(), 'Differenced Sales (d=1)')

print("Differencing removes the trend, making the series stationary.")
print("This is the 'I' (Integrated) part of ARIMA.")

## Section 2: Moving Averages & Smoothing

Before diving into models, **smoothing** helps us see the signal through the noise:

- **Simple Moving Average (SMA):** Average of last N values — smooths out short-term fluctuations
- **Exponential Moving Average (EMA):** Gives more weight to recent values — reacts faster to changes

These are useful for:
1. Visualization — see the underlying trend
2. Feature engineering — use as inputs to models
3. Simple forecasting — "tomorrow will look like the recent average"

In [ ]:
# Moving averages: smooth the noise to reveal the signal
fig, ax = plt.subplots(figsize=(14, 5))

# Plot original (semi-transparent)
ax.plot(ts.index, ts['sales'], alpha=0.3, color='gray', label='Original')

# Simple Moving Averages
ax.plot(ts.index, ts['sales'].rolling(7).mean(), label='7-day SMA', linewidth=1.5)
ax.plot(ts.index, ts['sales'].rolling(30).mean(), label='30-day SMA', linewidth=2)

# Exponential Moving Average (reacts faster to recent changes)
ax.plot(ts.index, ts['sales'].ewm(span=30).mean(), label='30-day EMA', 
        linewidth=2, linestyle='--')

ax.set_title('Moving Averages: Smoothing the Noise')
ax.set_ylabel('Sales ($)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print("Notice: SMA is smoother but lags behind. EMA reacts faster to changes.")
print("The 30-day SMA clearly reveals the yearly seasonal pattern.")

## Section 3: ARIMA (AutoRegressive Integrated Moving Average)

ARIMA is the workhorse of classical time series forecasting. It combines three components:

| Component | What it does | Parameter |
|-----------|-------------|-----------|
| **AR** (AutoRegressive) | Uses past values as predictors: $y_t = c + \phi_1 y_{t-1} + ... + \phi_p y_{t-p}$ | `p` = number of lags |
| **I** (Integrated) | Differences the series to make it stationary | `d` = number of differences |
| **MA** (Moving Average) | Uses past forecast *errors*: $y_t = c + \theta_1 \epsilon_{t-1} + ... + \theta_q \epsilon_{t-q}$ | `q` = number of error lags |

**How to choose (p, d, q)?**
1. **d:** Use the ADF test. If not stationary, d=1. Still not stationary after differencing? d=2.
2. **p:** Look at the **PACF** (Partial Autocorrelation Function) — significant lags suggest AR order
3. **q:** Look at the **ACF** (Autocorrelation Function) — significant lags suggest MA order

Or just use `auto_arima` from `pmdarima` to search automatically.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Use monthly aggregated data for cleaner ARIMA demo
monthly = ts.resample('ME').mean()
train = monthly[:'2024-06']
test = monthly['2024-07':]

print(f"Train: {len(train)} months | Test: {len(test)} months")

# ACF and PACF plots to guide parameter selection
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train['sales'].diff().dropna(), ax=ax1, lags=20)
ax1.set_title('ACF of Differenced Series (helps choose q)')
plot_pacf(train['sales'].diff().dropna(), ax=ax2, lags=20)
ax2.set_title('PACF of Differenced Series (helps choose p)')
plt.tight_layout()
plt.show()

print("\nSignificant spikes in PACF suggest AR terms (p).")
print("Significant spikes in ACF suggest MA terms (q).")

In [ ]:
# Fit ARIMA(2,1,2) model
model = ARIMA(train['sales'], order=(2, 1, 2))
fitted = model.fit()
print(fitted.summary())

# Forecast into test period
forecast = fitted.forecast(steps=len(test))

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(train.index, train['sales'], label='Train', linewidth=2)
plt.plot(test.index, test['sales'], label='Actual', marker='o', linewidth=2)
plt.plot(test.index, forecast.values, label='ARIMA(2,1,2) Forecast', 
         marker='s', linestyle='--', linewidth=2)
plt.fill_between(test.index, forecast.values * 0.95, forecast.values * 1.05, 
                 alpha=0.2, color='green', label='~95% band')
plt.legend(fontsize=11)
plt.title('ARIMA Forecast vs Actual')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.show()

# Calculate error
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(test['sales'], forecast.values)
print(f"\nForecast MAE: ${mae:.2f}")

## Section 4: Facebook Prophet

**Prophet** is designed specifically for business time series forecasting. Key advantages:

- **Handles missing data** and outliers gracefully
- **Built-in holiday effects** — just pass a holiday calendar
- **Automatic changepoint detection** — finds where trends shift
- **Additive or multiplicative seasonality** — configurable
- **Intuitive API** — fit/predict like scikit-learn

Prophet decomposes the series as: $y(t) = g(t) + s(t) + h(t) + \epsilon_t$

Where: $g(t)$ = trend, $s(t)$ = seasonality, $h(t)$ = holidays, $\epsilon_t$ = noise

In [ ]:
try:
    from prophet import Prophet
    
    # Prophet requires columns named 'ds' (date) and 'y' (value)
    df_prophet = ts.reset_index().rename(columns={'date': 'ds', 'sales': 'y'})
    train_p = df_prophet[df_prophet['ds'] < '2024-07-01']
    
    # Fit Prophet model
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, 
                daily_seasonality=False)
    m.fit(train_p)
    
    # Forecast 180 days into the future
    future = m.make_future_dataframe(periods=180)
    forecast_p = m.predict(future)
    
    # Plot forecast
    fig = m.plot(forecast_p)
    plt.title('Prophet Forecast (blue line = prediction, dots = actual)')
    plt.ylabel('Sales ($)')
    plt.show()
    
    # Plot components (trend + weekly + yearly patterns)
    fig2 = m.plot_components(forecast_p)
    plt.show()
    
    print("Prophet automatically detected trend changes and seasonal patterns!")
    
except ImportError:
    print("Prophet not installed. To install: pip install prophet")
    print("Skipping Prophet demo — the concepts still apply:")
    print("  - Prophet automates trend + seasonality decomposition")
    print("  - Great for business data with holidays and irregular gaps")

### ARIMA vs Prophet — When to Use Which?

| | ARIMA | Prophet |
|---|---|---|
| **Best for** | Simple, well-behaved series | Business data with holidays/gaps |
| **Setup** | Need to choose (p,d,q) carefully | Nearly automatic |
| **Missing data** | Cannot handle gaps | Handles gracefully |
| **Interpretability** | Model coefficients | Visual component plots |
| **Seasonality** | SARIMA for seasonal (more complex) | Built-in, multiple seasonalities |
| **Speed** | Very fast | Slower (uses Stan/MCMC) |

**Rule of thumb:** Start with Prophet for business data. Use ARIMA when you need fine-grained control or have very simple series.

---
# PART B: CLUSTERING & DIMENSIONALITY REDUCTION

## Section 5: Unsupervised Learning Overview

In supervised learning, we have labels (y). In **unsupervised learning**, we have NO labels — we must discover structure on our own.

**Two main tasks:**

| Task | Goal | Methods |
|------|------|---------|
| **Clustering** | Group similar items together | K-Means, DBSCAN, Hierarchical |
| **Dimensionality Reduction** | Simplify data while preserving structure | PCA, t-SNE, UMAP |

**Real-world applications:**
- **Customer segmentation** — who are our different customer types?
- **Anomaly detection** — which data points don't belong to any cluster?
- **Visualization** — reduce 50 features to 2D for plotting
- **Preprocessing** — PCA before clustering reduces noise

In [ ]:
# Generate synthetic customer data (RFM: Recency, Frequency, Monetary)
np.random.seed(42)
n_customers = 500

# 4 hidden segments with different RFM profiles
segments = np.random.choice([0, 1, 2, 3], n_customers, p=[0.3, 0.25, 0.25, 0.2])

recency = np.where(segments == 0, np.random.normal(10, 5, n_customers),
          np.where(segments == 1, np.random.normal(50, 15, n_customers),
          np.where(segments == 2, np.random.normal(100, 20, n_customers),
                   np.random.normal(200, 30, n_customers)))).clip(1)

frequency = np.where(segments == 0, np.random.normal(20, 5, n_customers),
            np.where(segments == 1, np.random.normal(10, 3, n_customers),
            np.where(segments == 2, np.random.normal(5, 2, n_customers),
                     np.random.normal(2, 1, n_customers)))).clip(1).astype(int)

monetary = np.where(segments == 0, np.random.normal(500, 100, n_customers),
           np.where(segments == 1, np.random.normal(200, 50, n_customers),
           np.where(segments == 2, np.random.normal(100, 30, n_customers),
                    np.random.normal(50, 20, n_customers)))).clip(10).round(2)

customers = pd.DataFrame({
    'recency': recency.round(0),
    'frequency': frequency,
    'monetary': monetary,
    'true_segment': segments  # we'll hide this and try to recover it
})

print("Customer Data (RFM):")
print(customers.describe().round(1))
print(f"\nTrue segment distribution:\n{customers['true_segment'].value_counts().sort_index()}")

In [ ]:
# Visualize the raw customer data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['recency', 'frequency', 'monetary']):
    axes[i].hist(customers[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{col.capitalize()} Distribution')
    axes[i].set_xlabel(col.capitalize())

plt.suptitle('RFM Feature Distributions (before scaling)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice the very different scales — recency goes to 200+, monetary to 500+")
print("We MUST scale before clustering, or recency would dominate the distance metric.")

## Section 6: K-Means Clustering

**Algorithm (Lloyd's):**
1. Randomly initialize K centroids
2. **Assign** each point to the nearest centroid
3. **Update** each centroid to the mean of its assigned points
4. Repeat steps 2-3 until convergence

**Choosing K:**
- **Elbow method:** Plot inertia (within-cluster sum of squares) vs K. Look for the "elbow" where adding more clusters stops helping much.
- **Silhouette score:** Measures how similar a point is to its own cluster vs neighboring clusters. Range: -1 to 1 (higher = better).

**Limitations:**
- Assumes spherical, equally-sized clusters
- Sensitive to initialization (use `n_init=10` or `kmeans++`)
- Must specify K in advance
- Sensitive to outliers

In [ ]:
# Step 1: ALWAYS scale before clustering
features = ['recency', 'frequency', 'monetary']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers[features])

print("Before scaling:")
print(f"  Recency  range: {customers['recency'].min():.0f} to {customers['recency'].max():.0f}")
print(f"  Monetary range: {customers['monetary'].min():.0f} to {customers['monetary'].max():.0f}")
print(f"\nAfter scaling (mean=0, std=1):")
print(f"  Recency  range: {X_scaled[:,0].min():.2f} to {X_scaled[:,0].max():.2f}")
print(f"  Monetary range: {X_scaled[:,2].min():.2f} to {X_scaled[:,2].max():.2f}")

In [ ]:
# Step 2: Find the best K using elbow method + silhouette score
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (within-cluster SS)')
ax1.set_title('Elbow Method')
ax1.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4')
ax1.legend()

# Silhouette plot
ax2.plot(K_range, silhouettes, 'rs-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score (higher = better)')
ax2.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4')
ax2.legend()

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Best K by silhouette score: {best_k}")
print(f"True number of segments: 4")

In [ ]:
# Step 3: Fit K-Means with K=4 and visualize
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customers['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

# Visualize in 2D (recency vs monetary, colored by cluster)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

# K-Means clusters
for c in sorted(customers['kmeans_cluster'].unique()):
    mask = customers['kmeans_cluster'] == c
    ax1.scatter(customers.loc[mask, 'recency'], customers.loc[mask, 'monetary'],
               c=colors[c], label=f'Cluster {c}', alpha=0.6, s=30)
ax1.set_xlabel('Recency (days)')
ax1.set_ylabel('Monetary ($)')
ax1.set_title('K-Means Clusters (K=4)')
ax1.legend()

# True segments for comparison
for s in sorted(customers['true_segment'].unique()):
    mask = customers['true_segment'] == s
    ax2.scatter(customers.loc[mask, 'recency'], customers.loc[mask, 'monetary'],
               c=colors[s], label=f'Segment {s}', alpha=0.6, s=30)
ax2.set_xlabel('Recency (days)')
ax2.set_ylabel('Monetary ($)')
ax2.set_title('True Segments (ground truth)')
ax2.legend()

plt.tight_layout()
plt.show()

sil = silhouette_score(X_scaled, customers['kmeans_cluster'])
print(f"K-Means silhouette score: {sil:.3f}")

## Section 7: DBSCAN (Density-Based Spatial Clustering)

**Key idea:** Clusters are dense regions separated by sparse regions. No need to specify K!

**Algorithm:**
1. For each point, find all neighbors within distance `eps`
2. If a point has at least `min_samples` neighbors, it's a **core point**
3. Core points and their neighbors form clusters
4. Points that aren't reachable from any core point are **outliers** (label = -1)

**Advantages over K-Means:**
- Finds clusters of arbitrary shape (not just spheres)
- Automatically detects outliers
- No need to specify number of clusters

**Key parameters:**
- `eps` — maximum distance between neighbors (too small = everything is noise, too large = one big cluster)
- `min_samples` — minimum points to form a dense region (rule of thumb: 2 * n_features)

In [ ]:
# Fit DBSCAN — need to tune eps carefully
from sklearn.neighbors import NearestNeighbors

# Use k-distance plot to find a good eps
nn = NearestNeighbors(n_neighbors=6)  # 2 * n_features = 6
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 4))
plt.plot(k_distances)
plt.xlabel('Points (sorted by distance)')
plt.ylabel('6th Nearest Neighbor Distance')
plt.title('K-Distance Plot (look for the "elbow" to choose eps)')
plt.axhline(y=0.8, color='red', linestyle='--', label='eps = 0.8')
plt.legend()
plt.tight_layout()
plt.show()

print("The elbow in the k-distance plot suggests a good eps value.")

In [ ]:
# Fit DBSCAN and compare with K-Means
dbscan = DBSCAN(eps=0.8, min_samples=6)
customers['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(customers['dbscan_cluster'])) - (1 if -1 in customers['dbscan_cluster'].values else 0)
n_outliers = (customers['dbscan_cluster'] == -1).sum()

print(f"DBSCAN found {n_clusters_db} clusters and {n_outliers} outliers")
print(f"Cluster distribution:\n{customers['dbscan_cluster'].value_counts().sort_index()}")

# Compare K-Means vs DBSCAN
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# K-Means
scatter1 = ax1.scatter(customers['recency'], customers['monetary'], 
                       c=customers['kmeans_cluster'], cmap='Set1', alpha=0.6, s=30)
ax1.set_title('K-Means (K=4)')
ax1.set_xlabel('Recency')
ax1.set_ylabel('Monetary')

# DBSCAN (outliers in black)
colors_db = customers['dbscan_cluster'].copy()
scatter2 = ax2.scatter(customers['recency'], customers['monetary'],
                       c=colors_db, cmap='Set1', alpha=0.6, s=30)
# Highlight outliers
outlier_mask = customers['dbscan_cluster'] == -1
ax2.scatter(customers.loc[outlier_mask, 'recency'], 
           customers.loc[outlier_mask, 'monetary'],
           c='black', marker='x', s=50, label=f'Outliers ({n_outliers})')
ax2.set_title(f'DBSCAN ({n_clusters_db} clusters)')
ax2.set_xlabel('Recency')
ax2.set_ylabel('Monetary')
ax2.legend()

plt.tight_layout()
plt.show()

## Section 8: PCA (Principal Component Analysis)

**Goal:** Reduce dimensionality while preserving as much variance as possible.

**How it works:**
1. Find the direction of maximum variance in the data (PC1)
2. Find the next direction of maximum variance, orthogonal to PC1 (PC2)
3. Continue for all dimensions
4. Keep only the top K components

**Why use PCA?**
- **Visualization:** Project high-dimensional data to 2D for plotting
- **Noise reduction:** Discard low-variance components (likely noise)
- **Speed:** Fewer features = faster training for downstream models
- **Decorrelation:** PCA components are uncorrelated (helps some algorithms)

**Key metric:** `explained_variance_ratio_` tells you how much information each component captures.

In [ ]:
# Fit PCA on the scaled customer data
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Scree plot: how much variance does each component explain?
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
ax1.bar(range(1, len(pca.explained_variance_ratio_) + 1), 
        pca.explained_variance_ratio_, color='steelblue', edgecolor='black')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot')
ax1.set_xticks(range(1, len(pca.explained_variance_ratio_) + 1))

for i, v in enumerate(pca.explained_variance_ratio_):
    ax1.text(i + 1, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Cumulative explained variance
cumulative = np.cumsum(pca.explained_variance_ratio_)
ax2.plot(range(1, len(cumulative) + 1), cumulative, 'ro-', linewidth=2, markersize=10)
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Variance Explained')
ax2.set_xticks(range(1, len(cumulative) + 1))
ax2.axhline(y=0.95, color='gray', linestyle='--', label='95% threshold')
ax2.legend()
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("Component loadings (how much each feature contributes to each PC):")
loadings = pd.DataFrame(pca.components_, columns=features,
                        index=[f'PC{i+1}' for i in range(len(features))])
print(loadings.round(3))

In [ ]:
# Visualize clusters in PCA space (2D projection)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

# K-Means clusters in PCA space
for c in sorted(customers['kmeans_cluster'].unique()):
    mask = customers['kmeans_cluster'] == c
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=colors[c], label=f'Cluster {c}', alpha=0.6, s=30)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax1.set_title('K-Means Clusters in PCA Space')
ax1.legend()

# True segments in PCA space
for s in sorted(customers['true_segment'].unique()):
    mask = customers['true_segment'] == s
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[s], label=f'Segment {s}', alpha=0.6, s=30)
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax2.set_title('True Segments in PCA Space')
ax2.legend()

plt.tight_layout()
plt.show()

print("PCA projects 3D data into 2D while preserving cluster structure.")
print("Compare the two plots — K-Means does a good job recovering the true segments!")

## Section 9: Putting It All Together — RFM Segmentation Pipeline

A complete customer segmentation workflow:
1. **Scale** the RFM features
2. **PCA** for visualization (optional noise reduction)
3. **K-Means** to cluster customers
4. **Profile** each cluster by its average RFM values
5. **Name** the segments to make them actionable for the business

In [ ]:
# Create segment profiles: mean RFM values per K-Means cluster
profiles = customers.groupby('kmeans_cluster')[features].mean().round(1)

# Name segments based on their RFM characteristics
# Low recency + high frequency + high monetary = Champions
# Sort by monetary to assign names
profiles_sorted = profiles.sort_values('monetary', ascending=False)
segment_names = {}
names = ['Champions', 'Loyal', 'At Risk', 'Lost']
for i, idx in enumerate(profiles_sorted.index):
    segment_names[idx] = names[i]

profiles['segment_name'] = profiles.index.map(segment_names)
print("Segment Profiles:")
print("=" * 60)
print(profiles)
print()

# How many customers in each segment?
customers['segment_name'] = customers['kmeans_cluster'].map(segment_names)
print("Segment Sizes:")
print(customers['segment_name'].value_counts())

In [ ]:
# Visualize segment profiles as a grouped bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

segment_order = ['Champions', 'Loyal', 'At Risk', 'Lost']
colors_seg = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

for i, (col, title) in enumerate(zip(features, 
    ['Recency (lower = better)', 'Frequency (higher = better)', 'Monetary (higher = better)'])):
    
    vals = [profiles.loc[profiles['segment_name'] == s, col].values[0] for s in segment_order]
    bars = axes[i].bar(segment_order, vals, color=colors_seg, edgecolor='black')
    axes[i].set_title(title, fontsize=11)
    axes[i].set_ylabel(col.capitalize())
    
    # Add value labels
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                    f'{val:.0f}', ha='center', fontweight='bold')

plt.suptitle('Customer Segment Profiles (RFM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Business actions:")
print("  Champions  → Reward and retain (loyalty programs)")
print("  Loyal      → Upsell and cross-sell")
print("  At Risk    → Win-back campaigns, special offers")
print("  Lost       → Low-cost re-engagement or accept churn")

In [ ]:
# Radar/spider chart for segment profiles
from matplotlib.patches import FancyBboxPatch

# Normalize profiles to 0-1 range for radar chart
profile_vals = profiles[features].copy()
for col in features:
    profile_vals[col] = (profile_vals[col] - profile_vals[col].min()) / \
                        (profile_vals[col].max() - profile_vals[col].min())
# Invert recency (lower is better)
profile_vals['recency'] = 1 - profile_vals['recency']

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for idx, name in segment_names.items():
    values = profile_vals.loc[idx].values.tolist()
    values += values[:1]
    color = colors_seg[segment_order.index(name)]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Recency\n(inverted)', 'Frequency', 'Monetary'], fontsize=12)
ax.set_title('Segment Radar Chart\n(higher = better customer)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

---
## Key Takeaways

### Time Series
- **Always decompose first** — understand trend, seasonality, and residual before modeling
- **Check stationarity** with the ADF test; use differencing if needed
- **ARIMA** for simple, well-behaved series where you want fine control over (p,d,q)
- **Prophet** for business time series with holidays, missing data, and trend changes
- **Moving averages** are useful for visualization and as features

### Clustering & PCA
- **Always scale before clustering** — StandardScaler is the default choice
- **Elbow + silhouette** together to choose K for K-Means
- **DBSCAN** when you expect non-spherical clusters or need outlier detection
- **PCA** for visualization (project to 2D) and noise reduction
- **Profile your clusters** — give them business-meaningful names to drive action

### The Pipeline
```
Raw Data → Scale → (Optional: PCA) → Cluster → Profile → Name → Act
```

---
## Exercises

### Exercise 1: ARIMA Model Comparison
Forecast the monthly data with ARIMA(1,1,1) vs ARIMA(2,1,2). Which has lower MAE on the test set?

In [ ]:
# Exercise 1: Compare ARIMA(1,1,1) vs ARIMA(2,1,2)
# Hint: fit both models on 'train', forecast len(test) steps, compute MAE

# YOUR CODE HERE
# model_111 = ARIMA(train['sales'], order=(1, 1, 1))
# fitted_111 = model_111.fit()
# forecast_111 = fitted_111.forecast(steps=len(test))
# mae_111 = mean_absolute_error(test['sales'], forecast_111.values)
#
# model_212 = ARIMA(train['sales'], order=(2, 1, 2))
# fitted_212 = model_212.fit()
# forecast_212 = fitted_212.forecast(steps=len(test))
# mae_212 = mean_absolute_error(test['sales'], forecast_212.values)
#
# print(f"ARIMA(1,1,1) MAE: ${mae_111:.2f}")
# print(f"ARIMA(2,1,2) MAE: ${mae_212:.2f}")
# print(f"Winner: {'ARIMA(1,1,1)' if mae_111 < mae_212 else 'ARIMA(2,1,2)'}")

### Exercise 2: K-Means — Choosing K
Try K-Means with K=3 and K=5 on the customer data. Which K has the best silhouette score? How do the segment profiles change?

In [ ]:
# Exercise 2: Compare K=3 vs K=5
# Hint: fit KMeans with each K, compute silhouette_score, print profiles

# YOUR CODE HERE
# for k in [3, 5]:
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(X_scaled)
#     sil = silhouette_score(X_scaled, labels)
#     print(f"K={k}: silhouette = {sil:.3f}")
#     
#     # Show profiles
#     temp = customers[features].copy()
#     temp['cluster'] = labels
#     print(temp.groupby('cluster')[features].mean().round(1))
#     print()

### Exercise 3: PCA + DBSCAN
Apply PCA to reduce the customer data to 2D, then run DBSCAN on the PCA-transformed data. Do the clusters look cleaner in PCA space vs. original space?

In [ ]:
# Exercise 3: PCA → DBSCAN
# Hint: use X_pca[:, :2] as input to DBSCAN, try different eps values

# YOUR CODE HERE
# pca_2d = PCA(n_components=2)
# X_pca2 = pca_2d.fit_transform(X_scaled)
#
# db_pca = DBSCAN(eps=0.7, min_samples=6)
# labels_pca = db_pca.fit_predict(X_pca2)
#
# n_clusters = len(set(labels_pca)) - (1 if -1 in labels_pca else 0)
# n_noise = (labels_pca == -1).sum()
# print(f"DBSCAN on PCA: {n_clusters} clusters, {n_noise} outliers")
#
# plt.figure(figsize=(8, 6))
# plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=labels_pca, cmap='Set1', alpha=0.6, s=30)
# plt.xlabel('PC1')
# plt.ylabel('PC2')
# plt.title('DBSCAN on PCA-reduced Data')
# plt.colorbar(label='Cluster')
# plt.show()